# Day 64：Capstone 收官日 — 毕业设计 Docker 部署上线、演示脚本与完整项目文档

> **🎙️ 讲师旁白**：欢迎来到《智枢企业级智能体平台》第六阶段最终章——**Day 64（`v1.0.0-day64`）**！今天我们将运行 `release_packager.py` 完成生产级发布预检、生成带 SHA-256 校验码的制品清单，并通过 `DemoRunner` 实测 5 步毕业答辩现场演示脚本：
> 1. **容器与环境预检**：校验多阶段 `Dockerfile`、`docker-compose.yml` 与 `.env.example`；
> 2. **安全拦截演练**：验证生产环境缺失密钥与占位符密钥的自动拦截；
> 3. **发布制品清单**：生成 `release_manifest.json`；
> 4. **5 步上线演示**：`Health Check -> RAG Query -> Financial Ratio -> HITL Approval -> Audit Export`。

## 1. 加载 Day 64 模块并执行发布前全量预检（Preflight Checks）

In [1]:
import json
import sys
import tempfile
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent

day64_dir = repo_root / "project_iterations" / "day_64"
if str(day64_dir) not in sys.path:
    sys.path.insert(0, str(day64_dir))

from zhishu_agent.release_packager import (
    DemoRunner,
    DeploymentConfig,
    check_env_vars,
    generate_release_manifest,
    run_preflight_checks,
)

preflight = run_preflight_checks(day64_dir)
print("模块路径:", day64_dir)
print("预检总结果 (overall_passed):", preflight["overall_passed"])
print("Dockerfile 通过项:", preflight["dockerfile"]["passed_checks"])
print("Compose 通过项:", preflight["compose"]["passed_checks"])

模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_64
预检总结果 (overall_passed): True
Dockerfile 通过项: ['多阶段构建 (Multi-stage build)', '非 root 运行用户 (USER appuser)', '端口声明 (EXPOSE)', '容器健康检查 (HEALTHCHECK)', '启动入口 (CMD/ENTRYPOINT)']
Compose 通过项: ['services 服务定义', 'ports 端口映射', 'healthcheck 健康检查', 'restart 重启策略', '环境变量配置 (environment/env_file)', '数据卷挂载 (volumes)']


/home/antigravity/ab6539763/-AI-Agent/.venv/lib/python3.11/site-packages/fastapi/testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2. 安全红线演练：拦截缺失密钥与生产占位符

In [2]:
bad_prod_cfg = DeploymentConfig(
    env_vars={
        "APP_ENV": "production",
        "DATABASE_URL": "sqlite:///./data/zhishu_prod.db",
        "JWT_SECRET": "change-me-in-production",
    }
)
env_result = check_env_vars(bad_prod_cfg)
print("安全校验是否通过:", env_result.passed)
print("拦截原因:", env_result.errors)

安全校验是否通过: False
拦截原因: ['生产环境禁止使用占位符密钥: JWT_SECRET', '缺少必填环境变量: LLM_API_KEY']


## 3. 生成带 SHA-256 校验和的发布制品清单（Release Manifest）

In [3]:
with tempfile.TemporaryDirectory() as tmp_dir:
    manifest_path = Path(tmp_dir) / "release_manifest.json"
    manifest = generate_release_manifest(day64_dir, output_path=manifest_path)
    print(f"版本: {manifest['version']} | 预检通过: {manifest['preflight_passed']}")
    for item in manifest["artifacts"]:
        print(f" - {item['path']} ({item['size_bytes']} bytes) sha256={item['sha256'][:16]}...")

版本: v1.0.0-day64 | 预检通过: True
 - Dockerfile (1721 bytes) sha256=7beda445386aeab3...
 - docker-compose.yml (644 bytes) sha256=526e4b8023112a38...
 - .env.example (335 bytes) sha256=dbba46ca7205904b...
 - main.py (1933 bytes) sha256=8195180832c5121b...
 - zhishu_agent/__init__.py (632 bytes) sha256=05a7e2a65af6da37...
 - zhishu_agent/release_packager.py (18357 bytes) sha256=4fc1055aaedf6a0c...


## 4. 运行 5 步毕业答辩全流程上线演示（DemoRunner）

In [4]:
runner = DemoRunner()
steps = runner.run_all()
for step in steps:
    print(f"Step {step['step']}: {step['name']} -> passed={step['passed']} ({step['duration_ms']} ms)")

print("\n财务比率工具输出:", json.dumps(steps[2]["output"], ensure_ascii=False))
print("HITL 审批结果:", json.dumps(steps[3]["output"], ensure_ascii=False))
print("审计日志事件总数:", steps[4]["output"]["total_events"])

Step 1: 服务健康检查 (Health Check) -> passed=True (22.28 ms)
Step 2: 混合 RAG 财报与制度检索 -> passed=True (4.58 ms)
Step 3: 财务比率分析工具调用 (ROE / 净利率 / 资产负债率) -> passed=True (3.87 ms)
Step 4: HITL 高风险人工审批闭环 -> passed=True (7.68 ms)
Step 5: 全链路合规审计日志导出 -> passed=True (3.21 ms)

财务比率工具输出: {"company": "星辰数科", "net_profit_margin_pct": 16.41, "roe_pct": 20.0, "debt_to_asset_pct": 38.0}
HITL 审批结果: {"approval_id": "hitl-16a352c8", "action": "研发GPU集群扩容资金划拨", "amount_cny": 850000.0, "requester": "陈默", "status": "approved", "reviewer": "张工", "comment": "预算合规，批准执行"}
审计日志事件总数: 5
